In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib as mpl
from Bio import SeqIO
import os
import shutil
import statistics
import re
from scipy.stats import norm
import seaborn as sns

In [ ]:
# stats for HMM are claculated from whatever scored into the HMM, doesn't matter
def calculate_quartiles_and_filter(group, multiplier=3.5):
    Q1 = group['Score'].quantile(0.25)
    Q3 = group['Score'].quantile(0.75)
    IQR = Q3 - Q1
    median = group['Score'].median()
    lower_bound = Q1 - multiplier * IQR
    
    group['Q1'] = Q1
    group['Q3'] = Q3
    group['IQR'] = IQR
    group['Median'] = median
    group['LowerBound'] = lower_bound
    
    filtered_group = group[group['Score'] >= lower_bound]
    
    return filtered_group

confusion_df = pd.read_csv('grassHmmDatabase/grassfam1/confusionMatrix', delimiter='\t')

filtered_confusion_df = confusion_df.groupby('HmmHogHit').apply(calculate_quartiles_and_filter).reset_index(drop=True)

filtered_confusion_df.to_csv('grassHmmDatabase/grassfam1/filtered_confusion_df', sep='\t', index=False)

confusion_df = pd.read_csv('grassHmmDatabase/grassfam1/filtered_confusion_df', delimiter='\t')

disagree_df = confusion_df.groupby(['Hog', 'HmmHogHit']).size().reset_index(name='count')
disagree_df = disagree_df[disagree_df['Hog'] != disagree_df['HmmHogHit']]
disagree_df = disagree_df[['count', 'Hog', 'HmmHogHit']]
disagree_df = disagree_df.rename(columns={'count': 'count','Hog': 'hog','HmmHogHit': 'hmmHogHit'})
disagree_df.to_csv('grassHmmDatabase/grassfam1/disagreeMatrix', sep='\t', index=False)

hog_count = confusion_df['Hog'].value_counts()
disagree_df['hogSize'] = disagree_df['hog'].map(hog_count)
disagree_df['ratio'] = disagree_df['count'].values / disagree_df['hogSize']
disagree_df.rename(columns={'count': 'mismatchCount'}, inplace=True)
columns = ['hog', 'hmmHogHit', 'mismatchCount', 'hogSize', 'ratio']
disagree_df = disagree_df[columns]
disagree_df.to_csv('grassHmmDatabase/grassfam1/disagreeMatrix2', sep='\t', index=False)
disagree_df

How how many mismatches? plus total error percentage per HOG/HMM.
disagreeMatrix3

In [ ]:
df = pd.read_csv('grassHmmDatabase/grassfam1/disagreeMatrix2', sep='\t')
grouped = df.groupby(['hog', 'hogSize']).agg({'mismatchCount': 'sum'}).reset_index()
grouped['new_ratio'] = grouped['mismatchCount'] / grouped['hogSize']
grouped['hmmWrongCount'] = df.groupby(['hog', 'hogSize']).size().values
grouped = grouped.rename(columns={'new_ratio': 'ratio', 'mismatchCount': 'mismatchCount', 'hogSize': 'hogSize', 'hmmWrongCount': 'hmmWrongCount', 'hog': 'hog'})
grouped = grouped[['hog', 'hmmWrongCount', 'mismatchCount', 'hogSize', 'ratio']]
grouped = grouped.sort_values(by='mismatchCount', ascending=False)
grouped.to_csv('grassHmmDatabase/grassfam1/disagreeMatrix3', sep='\t', index=False)
grouped

In [ ]:
output_table = 'grassHmmDatabase/grassfam1/geneInfo'
hog_directory = 'hogDirectoryFiltered'

with open(output_table, 'w') as f_out:
    # Write the header row
    f_out.write("hog\tgene\tgene_length\n")

    # Iterate through files in the MSA directory
    for filename in os.listdir(hog_directory):
        if filename.endswith(".fasta"):
            hog_id = filename.split('.')[1]
            hog_file = os.path.join(hog_directory, filename)

            species = set()
            lengths = []
            for record in SeqIO.parse(hog_file, "fasta"):
                gene_length = len(record.seq)
                gene = record.id
                f_out.write(f"{hog_id}\t{gene}\t{gene_length}\n")

Do joins to make a mega table 
'superMatrix'

In [ ]:
confusionMatrix = pd.read_csv('grassHmmDatabase/grassfam1/filtered_confusion_df', delimiter='\t')
geneInfo = pd.read_csv('grassHmmDatabase/grassfam1/geneInfo', delimiter='\t')
disagreeMatrix3 = pd.read_csv('grassHmmDatabase/grassfam1/disagreeMatrix3', delimiter='\t')
hogMsaStats = pd.read_csv('hogMsaStats.tsv', delimiter='\t')

merged_df = pd.merge(confusionMatrix, geneInfo, left_on='Gene', right_on='gene')
merged_df2 = pd.merge(merged_df, disagreeMatrix3, left_on='Hog', right_on='hog', how='left')
final_df = pd.merge(merged_df2, hogMsaStats, left_on='Hog', right_on='HOG', how='left')

final_df.drop(['gene', 'hog_x', 'hog_y', 'HOG', 'listOfLengths', 'min', 'max', 'species_count', 'gene_count'], axis=1, inplace=True)
superMatrix = final_df

gene_length = superMatrix['gene_length']
mean = superMatrix['mean'] #.round(2)
sd = superMatrix['sd'] #.round(2)

superMatrix['z_scores'] = ((gene_length - mean) / sd) #.round(2)
#superMatrix['median'] = superMatrix['median'].round(2)
z_scores = superMatrix['z_scores']

final_df.to_csv('grassHmmDatabase/grassfam1/superMatrix', sep='\t', index=False)
final_df

wrongMatrix = superMatrix[superMatrix['Hog'] != superMatrix['HmmHogHit']]
wrongMatrix.to_csv('grassHmmDatabase/grassfam1/wrongMatrix', sep='\t', index=False)

In [ ]:
def calculate_logoutliers(group):
    Q1 = group['logScore'].quantile(0.25)
    Q3 = group['logScore'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    group['logOutlier'] = group['logScore'] < lower_bound
    return group

superMatrix['logScore'] = np.log10(superMatrix['Score'])
meanLogScore = superMatrix.groupby('HmmHogHit')['logScore'].mean().sort_values()

superMatrix = superMatrix.groupby('HmmHogHit').apply(calculate_logoutliers).reset_index(drop=True)

superMatrix['HmmHogHit'] = pd.Categorical(superMatrix['HmmHogHit'], categories=meanLogScore.index, ordered=True)

In [ ]:
scores = confusion_df['Score']

num_bins = max(scores.astype(int)) - min(scores.astype(int)) // 2 + 1
plt.hist(confusion_df['Score'], bins=num_bins)
plt.title('Histogram of Scores')
plt.grid()
plt.show()

In [ ]:
filtered_df = confusion_df[confusion_df['Hog'] != confusion_df['HmmHogHit']]
scores = filtered_df['Score']

num_bins = max(scores.astype(int)) - min(scores.astype(int)) // 2 + 1
plt.hist(filtered_df['Score'], bins=num_bins)
# plt.xlim(left=0, right=4000)
# plt.ylim(bottom=0, top=300)
plt.title('Score of erroneous hmmHits')
plt.grid()
plt.show()

In [ ]:
filtered_df2 = confusion_df[confusion_df['Hog'] == confusion_df['HmmHogHit']]
scores = filtered_df2['Score']

num_bins = max(scores.astype(int)) - min(scores.astype(int)) // 2 + 1
plt.hist(filtered_df2['Score'], bins=num_bins, label='Consensus Hit Data')
plt.hist(filtered_df['Score'], bins=num_bins, color="red", label='Erroneous Hit Data')
# plt.xlim(left=0, right=4000)
# plt.ylim(bottom=0, top=300)
plt.title('Histogram of Score of Gene hit on HMM library')
# plt.ylim(0, 325)
plt.legend()
plt.grid()
plt.show()

In [ ]:
medianScore = np.log10(superMatrix.groupby('HmmHogHit')['Score'].median())
bottomQuartileScore = superMatrix.groupby('HmmHogHit')['Score'].quantile(.25)
topQuartileScore = superMatrix.groupby('HmmHogHit')['Score'].quantile(.75)

iqr = np.log10(topQuartileScore - bottomQuartileScore + 1)

m, b = np.polyfit(medianScore, iqr, deg=1)

print(m)
print(b)

plt.plot(medianScore, m*medianScore + b)
plt.title('IQR vs Median Score')
plt.xlabel('Median Score')
plt.ylabel('IQR')
plt.scatter(medianScore, iqr, s=.1, alpha=0.3)

In [ ]:
plt.scatter(final_df["median"].astype(int), final_df["mean"].astype(int), s = 1, label="Consensus data")

plt.scatter(wrongMatrix["median"].astype(int), wrongMatrix["mean"].astype(int), c="red",s=1, label="Erroneous data")
plt.xlabel('Log Median')
plt.ylabel('Log Mean')
plt.xscale("log")
plt.yscale("log")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.scatter(final_df["median"].astype(int), final_df["gene_length"].astype(int), s = 1, label="Consensus data")

plt.scatter(wrongMatrix["median"].astype(int), wrongMatrix["gene_length"].astype(int), c="red",s=1, label="Erroneous data")
plt.xlabel('Log Median')
plt.ylabel('Log gene_length')
plt.xscale("log")
plt.yscale("log")
plt.legend()
plt.grid()
plt.show()

In [ ]:
#plt.scatter(final_df["median"].astype(int), final_df["gene_length"].astype(int), s = 1)

plt.scatter(wrongMatrix["hogSize"].astype(int), wrongMatrix["mismatchCount"].astype(int), c="blue",s=2)
plt.title('')
plt.xlabel('Hog gene count')
plt.ylabel('Hog HMM disagree gene count')
#plt.xscale("log")
#plt.yscale("log")
plt.xlim(left=0)
plt.ylim(bottom=0)
plt.grid()
plt.show()

In [ ]:
Hog = wrongMatrix["Hog"]
Hmm = wrongMatrix["HmmHogHit"]

hogSet = set(Hog)
hmmSet = set(Hmm)

intersect = hogSet.intersection(hmmSet)

filteredWrongMatrix = wrongMatrix[wrongMatrix["Hog"].isin(intersect)]
filteredWrongMatrix2 = wrongMatrix[wrongMatrix["HmmHogHit"].isin(intersect)]
filteredWrongMatrix3 = pd.concat([filteredWrongMatrix, filteredWrongMatrix2])
filteredWrongMatrix3.to_csv('grassHmmDatabase/grassfam1/filteredWrongMatrix3', sep='\t', index=False)


[len(intersect), len(hogSet), len(hmmSet)]

In [ ]:
G = nx.DiGraph()

wrongMatrix = pd.read_csv('grassHmmDatabase/grassfam1/wrongMatrix', sep="\t")

toGraph = wrongMatrix[['Hog', 'HmmHogHit']].drop_duplicates()

edges = toGraph[['Hog', 'HmmHogHit']].to_records(index=False)
G.add_edges_from(edges)

pos = nx.spring_layout(G)

nx.draw(G, with_labels=True, font_size=10, node_size=500)

plt.title('directed graph of incorrect HMM assignment')
plt.show()

In [ ]:
print(wrongMatrix.shape)

In [ ]:
G = nx.DiGraph()

toGraph = filteredWrongMatrix3[['Hog', 'HmmHogHit']].drop_duplicates()

nodes = toGraph['Hog'].unique()
G.add_nodes_from(nodes)
edges = toGraph[['Hog', 'HmmHogHit']].to_records(index=False)
G.add_edges_from(edges)
cycles = list(nx.simple_cycles(G))

# Create a subgraph containing only nodes and edges that are part of cycles
cyclic_graph = nx.DiGraph()
for cycle in cycles:
    cyclic_graph.add_nodes_from(cycle)
    cyclic_graph.add_edges_from(zip(cycle, cycle[1:] + [cycle[0]]))

# Calculate positions for the nodes
pos = nx.spring_layout(cyclic_graph)

# Draw the cyclic portion of the graph with smaller nodes
nx.draw(cyclic_graph, pos, with_labels=True, font_size=5, node_size=100, node_color="red")

plt.title('cyclic only directed graph of incorrect HMM assignment')
plt.show()

Histogram of Z-Scores for all

In [ ]:
z_scores = superMatrix['z_scores']

#set all infinite z scores to 0 sicne this happens when sd is 0, if sd is 0 all genes in hog same length so z_score is 0
z_scores_filtered = np.where(np.isfinite(z_scores), z_scores, 0)

mu, std = norm.fit(z_scores_filtered)

plt.hist(z_scores_filtered,density=True, bins= 100)
# xmin, xmax = plt.xlim()
# x = np.linspace(xmin, xmax, 100)
# p = norm.pdf(x, mu, std)
# plt.plot(x, p, 'k', linewidth=1)
# plt.xlim(-12.5,12.5)
# plt.ylim(0,0.825)
plt.title('Histogram of Z-Score of gene length in parent hog')
plt.xlabel('z score')
plt.ylabel('frequency')
plt.grid()
plt.show()

In [ ]:
plt.hist(wrongMatrix['z_scores'],density=True, bins= 100, color="red")
# xmin, xmax = plt.xlim()
# x = np.linspace(xmin, xmax, 100)
# p = norm.pdf(x, mu, std)
# plt.plot(x, p, 'k', linewidth=1)
# plt.xlim(-12.5,12.5)
# plt.ylim(0,0.825)
plt.title('Histogram of Z-Score of gene length of erroneous hits')
plt.xlabel('z score')
plt.ylabel('frequency')
plt.grid()
plt.show()

In [ ]:
plt.hist(disagree_df['ratio'], bins=100)
plt.title('Ratio of disagreement between HMM and HOG')
plt.grid()
plt.show()

In [ ]:
# sns.stripplot(data=superMatrix, x='Score', y='HmmHogHit', order=meanScore.index, hue='Outlier', 
#               palette={True: 'red', False: 'blue'}, size=.1, jitter=True, alpha=0.7)

# plt.xlabel('Score')
# plt.ylabel('HmmHogHit')
# plt.yticks([])
# plt.title('Score Distribution within HmmHogHit Groups (Sorted by Mean Score)')
# plt.legend(title='Outlier', loc='upper right')

# plt.show()

In [ ]:
# sns.stripplot(data=superMatrix, x='logScore', y='HmmHogHit', order=meanLogScore.index, hue='logOutlier', 
#               palette={True: 'red', False: 'blue'}, size=.1, jitter=True, alpha=0.7)

# plt.xlabel('logScore')
# plt.ylabel('HmmHogHit')
# plt.yticks([])
# plt.title('logScore Distribution within HmmHogHit Groups (Sorted by Mean logScore)')
# plt.legend(title='logOutlier', loc='upper right')

# plt.show()

In [ ]:
# print("number of log Outliers: ", sum(superMatrix["logOutlier"]))
# print("number of Outliers: ", sum(superMatrix["Outlier"]))
# print("number of genes total: ", len(superMatrix))

In [ ]:
# Make plot of number of outliers per HMM for both log and linear, X axis ordered by score Y value is number of outlier

# outlier_counts = superMatrix.groupby("HmmHogHit")["Outlier"].sum().reindex(meanScore.index)

# log_outlier_counts = superMatrix.groupby("HmmHogHit")["logOutlier"].sum().reindex(meanScore.index)

# plt.bar(outlier_counts.index, outlier_counts.values, alpha=0.7, label='Linear Scale Outliers')
# plt.xlabel('HmmHogHit (ordered by mean score)')
# plt.ylabel('Number of Outliers')
# plt.title('Number of Outliers per HMM (Linear Scale)')
# plt.xticks([])
# plt.tight_layout()
# plt.legend()
# plt.show()

# plt.bar(log_outlier_counts.index, log_outlier_counts.values, alpha=0.7, label='Log Scale Outliers')
# plt.xlabel('HmmHogHit (ordered by mean log score)')
# plt.ylabel('Number of Outliers')
# plt.title('Number of Outliers per HMM (Log Scale)')
# plt.xticks([])
# plt.tight_layout()
# plt.legend()
# plt.show()

In [ ]:
# print("num HMMs with log outliers: ", sum(log_outlier_counts.values > 0))
# print("num HMMs with outliers: ", sum(outlier_counts.values > 0))
# print("num HMMs: ", sum(outlier_counts.values > -1))

In [ ]:
# These scored to nothing

hogList = pd.read_csv('grassHmmDatabase/grassfam1/geneInfo', delimiter='\t')['gene']
geneInfo = pd.read_csv('grassHmmDatabase/grassfam1/geneInfo', delimiter='\t')
hmmList = pd.read_csv('grassHmmDatabase/grassfam1/filtered_confusion_df', delimiter='\t')['Gene']

hogSet = set(hogList)
hmmSet = set(hmmList)

unique_to_hog = hogSet - hmmSet

filtered_geneInfo = geneInfo[geneInfo['gene'].isin(unique_to_hog)]

print(filtered_geneInfo)
filtered_geneInfo.to_csv('grassHmmDatabase/grassfam1/dropped', sep='\t', index=False)

In [ ]:
disagreeMatrix3 = grouped
disagreeMatrix3
sum_under_90 = disagreeMatrix3[disagreeMatrix3['ratio'] >= .5 ]
sum_under_90
sum_under_90.sort_values('mismatchCount')
# sum_under_90.to_csv('under90', sep='\t', index=False)



In [ ]:
confusion_df = pd.read_csv('grassHmmDatabase/grassfam1/confusionMatrix', delimiter='\t')

# Make elbow plot of calculate_quartiles_and_filter using 6 - 1.5 * IQR as cutoff
# Make graph of 1.5 -> 5 times IQR for gene counts

lengths = []

cutoffs = [1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5]

for cutoff in cutoffs:
    filtered_df = confusion_df.groupby('HmmHogHit').apply(lambda x: calculate_quartiles_and_filter(x, cutoff)).reset_index(drop=True)
    lengths.append(len(filtered_df))

# Plot the results
plt.plot(cutoffs, lengths, marker='o')
plt.xlabel('IQR Multiplier (Cutoff)')
plt.ylabel('Number of Genes')
plt.title('Number of Genes Remaining by IQR Multiplier')
plt.grid(True)
plt.show()


In [ ]:
big = len(confusion_df)

outs = [(big - x) for x in lengths]

plt.plot(cutoffs, outs, marker='o')
plt.xlabel('IQR Multiplier (Cutoff)')
plt.ylabel('Number of Genes removed')
plt.title('Number of Genes Filtered by IQR Multiplier')
plt.grid(True)
plt.show()

In [ ]:
print(big)

In [ ]:
big = len(confusion_df)

outs = [(big - x)/big for x in lengths]

plt.plot(cutoffs, outs, marker='o')
plt.xlabel('IQR Multiplier (Cutoff)')
plt.ylabel('Number of Genes removed')
plt.title('Number of Genes Filtered by IQR Multiplier')
plt.grid(True)
plt.show()

In [ ]:
#increment the pwd suffix by 1 and make a new directory in the parent directory

pwd = os.getcwd().split('/')[-1]
match = re.search(r'(\d+)$', pwd)
suffix = int(match.group(1)) + 1

new_dir = '/'.join(os.getcwd().split('/')[:-1]) + '/' + re.sub(r'\d+$', str(suffix), pwd)

# make bashScripts directory in the new directory     
# Copy the .sh files from bashSciprts to the new directory
# Copy the .ipynb files to the new directory

try:
    os.mkdir(new_dir)
    os.mkdir(new_dir + '/bashScripts')
    os.mkdir(new_dir + '/bashScripts' + '/bashLogs')
except:
    pass

sh_files = [f for f in os.listdir('bashScripts') if f.endswith('.sh')]
ipynb_files = [f for f in os.listdir() if f.endswith('.ipynb')]

for f in sh_files:
    try:
        shutil.copy('bashScripts/' + f, new_dir + '/bashScripts')
    except:
        print("Error copying file: ", f)

for f in ipynb_files:
    try:
        shutil.copy(f, new_dir)
    except:
        print("Error copying file: ", f)
    